In [2]:
import torch 
import os
import pandas as pd
import random


def split_sample(datapath, ratio, seed=10, number=25000):
    # 按患者为单位划分数据集
    random.seed(seed)
    df = pd.read_csv(datapath)

    # 阳性阴性分组
    pos_patients = df[df["Target"] == 1]["patientId"].unique().tolist()
    neg_patients = df[df["Target"] == 0]["patientId"].unique().tolist()
    random.shuffle(pos_patients)
    random.shuffle(neg_patients)

    # 各取 number/2 个患者，保证正负均衡
    n_half = number // 2
    selected_pos = pos_patients[:min(n_half, len(pos_patients))]
    selected_neg = neg_patients[:min(n_half, len(neg_patients))]
    selected_all = selected_pos + selected_neg
    random.shuffle(selected_all)

    # 为每个患者聚合所有标记
    data_list = []
    target_data = []
    pos_set = set(selected_pos)

    for patient_id in selected_all:
        dcm_path = os.path.join("./dataset/raw-RSNA/pth_data/stage_2_train_images", f"{patient_id}.pth")
        img = torch.load(dcm_path, weights_only=True).get("img")

        if patient_id in pos_set:
            # 阳性患者
            patient_rows = df[(df["patientId"] == patient_id) & (df["Target"] == 1)]
            bboxes = []
            for _, row in patient_rows.iterrows():
                bboxes.append({
                    "x": row["x"],
                    "y": row["y"],
                    "width": row["width"],
                    "height": row["height"],
                })
            item_dct = {
                "patientId": patient_id,
                "Target": 1,
                "bboxes": bboxes,
                "img": img,
            }
            target_data.append(item_dct)

        else:
            # 阴性患者
            item_dct = {
                "patientId": patient_id,
                "Target": 0,
                "bboxes": [],
                "img": img,
            }

        data_list.append(item_dct)

    random.seed(seed)
    random.shuffle(data_list)

    total = len(data_list)
    train_end = int(total * ratio)

    train_set = data_list[:train_end]
    val_set = data_list[train_end:]

    n_pos = sum(1 for it in train_set if it["Target"] == 1)
    n_neg = sum(1 for it in train_set if it["Target"] == 0)
    total_bboxes = sum(len(it["bboxes"]) for it in train_set)
    print(f"Total: {total} | Train: {len(train_set)} (pos={n_pos}, neg={n_neg}, bboxes={total_bboxes}) | Val: {len(val_set)} |")

    all_data = {
        "train_data": train_set,
        "val_data": val_set,
    }

    return all_data, target_data


ratio = 0.8  # 训练集比例
all_data, target_data = split_sample("./dataset/raw-RSNA/stage_2_train_labels.csv", ratio)

Total: 18512 | Train: 14809 (pos=4829, neg=9980, bboxes=7684) | Val: 3703 |


In [3]:
import json
from torchvision.transforms.functional import to_pil_image


def get_annotations(datalist:list, idx_map:dict):
    # 获取 annotations 文件夹中 data.json 的 annotation 字段
    annotations = list()
    idx = 1
    for img_dct in datalist:
        if img_dct["bboxes"]:
            for bbox in img_dct["bboxes"]:
                bbox_norm = [bbox["x"], bbox["y"], bbox["width"], bbox["height"]]
                annotation = {
                    "id": idx,
                    "image_id": idx_map[img_dct.get("patientId")],
                    "category_id": 1,
                    "bbox": bbox_norm,
                    "area": bbox_norm[-1]*bbox_norm[-2],
                    "iscrowd": 0,
                }
                idx += 1
                annotations.append(annotation)
    
    return annotations
    

def get_images(datalist:list):
    # 获取 annotations 文件夹中 data.json 的 images 字段
    images = list()
    idx_map = dict()
    for idx, img_dict in enumerate(datalist):
        image = {
            "id": idx+1,
            "file_name": (img_dict.get("patientId")+".jpg"),
            "width": 1024,
            "height": 1024,
        }
        images.append(image)
        idx_map[img_dict.get("patientId")] = idx+1
    return images, idx_map



def convert2coco(all_data, savefold):
    os.makedirs(savefold, exist_ok=True)
    os.makedirs(f"{savefold}/annotations", exist_ok=True)
    os.makedirs(f"{savefold}/train2017", exist_ok=True)
    os.makedirs(f"{savefold}/val2017", exist_ok=True)
    
    for key in all_data:
        # annotations 部分 json 文件生成存储
        images, idxmap = get_images(all_data.get(key))
        annotations = get_annotations(all_data.get(key), idxmap)
        categories = [{"id": 1, "name": "pneumonia", "supercategory": "none"}]
        data_json = {
            "images": images,
            "annotations": annotations,
            "categories": categories,
        }
        savepath = f"{savefold}/annotations/{key}.json"
        with open(savepath, "w") as f:
            json.dump(data_json, f, indent=4)
        print(f"json file saved to {savepath}")

        # imgs 部分文件生成存储
        typefold = (key.split("_")[0]+"2017")
        basefold = f"{savefold}/{typefold}"

        for data_dict in all_data.get(key):
            img_tensor = data_dict.get("img")
            pil_img = to_pil_image(img_tensor).convert("RGB")
            imgname = (data_dict["patientId"]+".jpg")
            savepath = os.path.join(basefold, imgname)
            pil_img.save(savepath, quality=100)

        print(f"imgs saved to {basefold}")


convert2coco(all_data, "./dataset/LRSNA")

json file saved to ./dataset/LRSNA/annotations/train_data.json
imgs saved to ./dataset/LRSNA/train2017
json file saved to ./dataset/LRSNA/annotations/val_data.json
imgs saved to ./dataset/LRSNA/val2017
